In [ ]:
import json
import pandas as pd
import os
import glob

base_path = "AF3_Modeling/Dimer/Full_Data/"
json_files = glob.glob(os.path.join(base_path, "*.json"))

out_csv_all = os.path.join(base_path, "contacts_interchain_merged.csv")
all_dfs = []

for json_file in json_files:
    filename = os.path.basename(json_file)

    # Extract chain info from filename
    parts = filename.split("_")
    if len(parts) > 7:
        chainA_label = parts[6]
        chainB_label = parts[7]
    else:
        chainA_label, chainB_label = "chainA", "chainB"

    with open(json_file, "r") as f:
        data = json.load(f)

    if isinstance(data, list):
        if not data:
            continue
        data = data[0]

    contact_probs = data.get("contact_probs", [])
    token_chain_ids = data.get("token_chain_ids", [])
    token_res_ids = data.get("token_res_ids", [])

    if not isinstance(contact_probs, list) or not all(isinstance(row, list) for row in contact_probs):
        raise ValueError(f"'contact_probs' in {filename} should be a list of lists.")
    if len(token_chain_ids) != len(contact_probs) or len(token_res_ids) != len(contact_probs):
        raise ValueError(f"'token_chain_ids' / 'token_res_ids' length mismatch in {filename}.")

    # Mapping dict: ex) A -> o, B -> w1
    chain_map = {"A": chainA_label, "B": chainB_label}

    filtered_pairs = []
    for row_idx, row in enumerate(contact_probs):
        for col_idx, prob in enumerate(row):
            if prob is not None and prob > 0.0 and abs(row_idx - col_idx) > 5:
                chain_1 = token_chain_ids[row_idx]
                chain_2 = token_chain_ids[col_idx]

                if chain_1 != chain_2:  # inter-chain
                    mapped_chain1 = chain_map.get(chain_1, chain_1)
                    mapped_chain2 = chain_map.get(chain_2, chain_2)

                    # Take token_res_ids
                    res_1 = token_res_ids[row_idx]
                    res_2 = token_res_ids[col_idx]

                    filtered_pairs.append([res_1, mapped_chain1, res_2, mapped_chain2, prob, filename])

    df = pd.DataFrame(
        filtered_pairs,
        columns=["Residue 1", "Chain 1", "Residue 2", "Chain 2", "Probability", "SourceFile"]
    )
    all_dfs.append(df)

if all_dfs:
    merged_df = pd.concat(all_dfs, ignore_index=True)
    merged_df.to_csv(out_csv_all, index=False)
    print(f"Saved merged file: {out_csv_all}, total pairs: {len(merged_df)}")

    for threshold in [0.2, 0.5, 0.8]:
        subset = merged_df[merged_df["Probability"] >= threshold]
        out_csv_thr = os.path.join(base_path, f"contacts_interchain_merged_thr{threshold}.csv")
        subset.to_csv(out_csv_thr, index=False)
        print(f"Saved {out_csv_thr}, pairs: {len(subset)}")
else:
    print("No valid data found.")
